# MMFi 跨模态动量对比学习实验演示

本 notebook 展示如何使用 MMFi 数据集进行跨模态动量对比学习的训练和推理。

## 目录
1. 环境安装
2. 配置说明
3. 数据加载演示
4. 模型前向传播演示
5. 训练演示
6. 推理演示

## 1. 环境安装

```bash
# 创建虚拟环境
conda create -n csi_har python=3.8
conda activate csi_har

# 安装 PyTorch（根据 CUDA 版本选择）
pip install torch torchvision torchaudio

# 安装其他依赖
pip install numpy scipy matplotlib pyyaml scikit-learn tqdm seaborn
```

In [ ]:
# 导入必要的库
import sys
import os
sys.path.insert(0, '..')

import torch
import numpy as np
import yaml

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 设备: {torch.cuda.get_device_name(0)}")

## 2. 配置说明

配置文件位于 `configs/default.yaml`，主要包含：
- 数据集配置（路径、划分比例、协议）
- 模型配置（编码器维度、头部维度）
- 训练配置（学习率、批次大小、损失权重）
- CSI 增强配置

In [ ]:
# 加载配置
with open('../configs/default.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("配置内容:")
print(yaml.dump(config, default_flow_style=False, allow_unicode=True))

## 3. 数据加载演示

数据集返回格式：
```python
{
    "csi": torch.FloatTensor([T, Subcarriers, Antennas]),
    "skeleton": torch.FloatTensor([T, J, 3]),
    "label": int,
    "sample_id": str
}
```

In [ ]:
# 模拟数据加载（实际使用时需要真实数据）
# from datasets.mmfi_dataset import create_dataloaders
# train_loader, val_loader, num_classes = create_dataloaders(config)

# 模拟数据
batch_size = 4
T = 100  # 时间步
num_subcarriers = 30
num_antennas = 3
num_joints = 17

# 模拟一个 batch
batch = {
    'csi': torch.randn(batch_size, T, num_subcarriers, num_antennas),
    'skeleton': torch.randn(batch_size, T, num_joints, 3),
    'label': torch.randint(0, 13, (batch_size,)),
    'sample_id': [f'E01_S0{i}_A01' for i in range(1, batch_size + 1)]
}

print("数据形状:")
print(f"  CSI: {batch['csi'].shape}")
print(f"  Skeleton: {batch['skeleton'].shape}")
print(f"  Labels: {batch['label']}")
print(f"  Sample IDs: {batch['sample_id']}")

## 4. 模型前向传播演示

### 4.1 CSI 编码器（双流架构）

In [ ]:
from models.csi_encoder import CSIEncoder

# 创建 CSI 编码器
csi_encoder = CSIEncoder(
    num_subcarriers=num_subcarriers,
    num_antennas=num_antennas,
    channel_dim=128,
    temporal_dim=128,
    fusion_dim=256
)

# 前向传播
csi_input = batch['csi']
f_csi = csi_encoder(csi_input)

print(f"CSI 输入形状: {csi_input.shape}")
print(f"CSI 特征形状: {f_csi.shape}")
print(f"CSI Encoder 参数量: {sum(p.numel() for p in csi_encoder.parameters()):,}")

### 4.2 RGB 编码器（动量编码器）

In [ ]:
from models.rgb_encoder import RGBEncoder

# 创建 RGB 编码器
rgb_encoder = RGBEncoder(
    num_joints=num_joints,
    coord_dim=3,
    spatial_dim=128,
    temporal_dim=128,
    fusion_dim=256
)

# 验证参数已冻结
print(f"RGB Encoder 参数已冻结: {all(not p.requires_grad for p in rgb_encoder.parameters())}")

# 前向传播（必须使用 no_grad）
skeleton_input = batch['skeleton']
with torch.no_grad():
    f_rgb = rgb_encoder(skeleton_input)

print(f"Skeleton 输入形状: {skeleton_input.shape}")
print(f"RGB 特征形状: {f_rgb.shape}")

### 4.3 头部模块

In [ ]:
from models.heads import HeadsModule

# 创建头部模块
heads = HeadsModule(
    fusion_dim=256,
    projector_dim=128,
    classifier_hidden=256,
    regressor_hidden=256,
    num_classes=13,
    num_joints=num_joints
)

# 投影
z_csi = heads.project_csi(f_csi)
with torch.no_grad():
    z_rgb = heads.project_rgb(f_rgb)

# 分类
logits = heads.classify(f_csi)

# 回归
joints_pred = heads.regress(f_csi)

print(f"CSI 投影形状: {z_csi.shape}")
print(f"RGB 投影形状: {z_rgb.shape}")
print(f"分类 logits 形状: {logits.shape}")
print(f"骨架回归形状: {joints_pred.shape}")
print(f"\nL2 范数（应为 1）: {torch.norm(z_csi, dim=1)}")

## 5. 训练演示

### 5.1 单步训练

In [ ]:
from losses.supcon import CrossModalSupConLoss
from utils.queue import MemoryQueue
from models.momentum import momentum_update
import torch.nn as nn

# 损失函数
contrastive_loss = CrossModalSupConLoss(temperature=0.07)
classification_loss = nn.CrossEntropyLoss()
regression_loss = nn.L1Loss()

# 内存队列
queue = MemoryQueue(feature_dim=128, queue_size=64)

# 优化器（不包含 RGB 参数）
optimizer = torch.optim.AdamW([
    {'params': csi_encoder.parameters(), 'lr': 1e-4},
    {'params': heads.get_trainable_params(), 'lr': 1e-3}
], weight_decay=1e-4)

# 单步训练
csi_encoder.train()
heads.train()
rgb_encoder.eval()

# 1. CSI forward
f_csi = csi_encoder(batch['csi'])

# 2. RGB forward (no_grad)
with torch.no_grad():
    f_rgb = rgb_encoder(batch['skeleton'])

# 3. Project
z_csi = heads.project_csi(f_csi)
with torch.no_grad():
    z_rgb = heads.project_rgb(f_rgb)

# 4. Compute losses
labels = batch['label']
queue_keys = queue.get_queue() if not queue.is_empty() else None

loss_con = contrastive_loss(z_csi, z_rgb, labels, queue_keys)
loss_cls = classification_loss(heads.classify(f_csi), labels)

skeleton_pooled = batch['skeleton'].mean(dim=1).reshape(batch_size, -1)
loss_reg = regression_loss(heads.regress(f_csi), skeleton_pooled)

loss = 1.0 * loss_con + 1.0 * loss_cls + 0.5 * loss_reg

# 5. Backward
optimizer.zero_grad()
loss.backward()

# 6. Optimizer step
optimizer.step()

# 7. Momentum update
momentum_update(rgb_encoder, csi_encoder, m=0.999)

# 8. Queue enqueue
queue.enqueue(z_rgb.detach(), labels)

print(f"对比损失: {loss_con.item():.4f}")
print(f"分类损失: {loss_cls.item():.4f}")
print(f"回归损失: {loss_reg.item():.4f}")
print(f"总损失: {loss.item():.4f}")
print(f"队列大小: {queue.size()}")

## 6. 推理演示

**关键约束**：推理时只使用 CSI Encoder + Classifier

In [ ]:
# 推理模式
csi_encoder.eval()
heads.eval()

# 模拟推理输入
test_csi = torch.randn(1, T, num_subcarriers, num_antennas)

with torch.no_grad():
    # 只使用 CSI Encoder + Classifier
    f_csi = csi_encoder(test_csi)
    logits = heads.classify(f_csi)
    probs = torch.softmax(logits, dim=1)
    pred = logits.argmax(dim=1)

print(f"预测类别: {pred.item()}")
print(f"置信度: {probs.max().item():.4f}")
print(f"\n【推理约束验证】")
print(f"  - 未使用 RGB Encoder: ✓")
print(f"  - 未使用 Queue: ✓")
print(f"  - 未使用 Projector: ✓")
print(f"  - 未使用 Regressor: ✓")

## 7. 运行完整训练

```bash
# 训练
python train.py --config configs/default.yaml

# 评估
python evaluate.py --config configs/default.yaml --checkpoint checkpoints/best.pth

# 推理
python inference.py --config configs/default.yaml --checkpoint checkpoints/best.pth --input path/to/csi.npy
```